**Phase 1:** Extract Labels Locally

In [0]:
import zipfile
import os
import shutil

# Step 1: delete any partial extraction on the volume
dbutils.fs.rm("/Volumes/vehicle_project/bronze/raw_files/extracted/", recurse=True)

# Step 2: extract to local disk (fast), with progress updates
zip_path = "/Volumes/vehicle_project/bronze/raw_files/labels.zip"
local_extract_path = "/tmp/labels_extracted/"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    file_list = zip_ref.namelist()
    total_files = len(file_list)
    print(f"Total files to extract: {total_files}")

    for i, file in enumerate(file_list):
        zip_ref.extract(file, local_extract_path)
        if i % 1000 == 0 or i == total_files - 1:
            print(f"Extracted {i + 1} / {total_files} files...")

print("EXTRACTION COMPLETE")
print("Sample of extracted contents:", os.listdir(local_extract_path)[:10])

**Phase 2:** Consolidate Labels into One CSV

In [0]:
import os
import pandas as pd

labels_dir = "/tmp/labels_extracted/labels"
rows = []

for split in ["train", "valid", "test"]:
    split_dir = os.path.join(labels_dir, split)
    for fname in os.listdir(split_dir):
        with open(os.path.join(split_dir, fname)) as f:
            for line in f:
                class_id, x_center, y_center, width, height = line.strip().split()
                rows.append({
                    "filename": fname.replace(".txt", ".jpg"),
                    "class_id": int(class_id),
                    "x_center": float(x_center),
                    "y_center": float(y_center),
                    "width": float(width),
                    "height": float(height),
                    "split": split
                })

df = pd.DataFrame(rows)
print(f"Total rows: {len(df)}")

os.makedirs("/Volumes/vehicle_project/bronze/raw_files/annotations", exist_ok=True)
df.to_csv("/Volumes/vehicle_project/bronze/raw_files/annotations/all_annotations.csv", index=False)
print("Saved to volume.")